In [1]:
import os

os.environ["FLAGS_enable_pir_api"] = "0"

import pandas as pd
from paddleocr import PaddleOCR
import logging
import warnings
import gc


# --- THE C++ KERNEL FIX ---

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger('ppocr').setLevel(logging.ERROR)

def extract_street_typography_gpu(
    base_dir="E:/street_imagery_project/raw_images", 
    output_csv="E:/street_imagery_project/metadata/extracted_typography.csv"
):
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        processed_ids = set(existing_df['image_id'].astype(str).unique())
        print(f"Resuming run: Found {len(processed_ids)} already processed images in CSV.")
    else:
        processed_ids = set()
        pd.DataFrame(columns=["neighborhood", "image_id", "detected_text", "confidence_score"]).to_csv(output_csv, index=False, encoding='utf-8-sig')

    print("Initializing OCR Engine (Korean/English Model on GPU)...")
    ocr = PaddleOCR(use_textline_orientation=True, lang='korean')
    
    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

    for neighborhood in neighborhoods:
        print(f"\nScanning {neighborhood.title()}...")
        neighborhood_path = os.path.join(base_dir, neighborhood)
        
        image_paths = [
            os.path.join(neighborhood_path, f) for f in os.listdir(neighborhood_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
        ]
        
        batch_extracted = []
        total_images = len(image_paths)
        
        for i, img_path in enumerate(image_paths, 1):
            image_id = os.path.basename(img_path).replace(".png", "").replace(".jpg", "")
            
            if image_id in processed_ids or os.path.getsize(img_path) < 5000:
                continue

            try:
                result = ocr.predict(img_path)
                if not result:
                    continue

                for res in result:
                    texts, scores = [], []
                    
                    # --- THE FIXED V3 PARSER ---
                    # Attempt to extract keys directly from the custom PaddleX object
                    try:
                        texts = res['rec_texts']
                        scores = res['rec_scores']
                    except Exception:
                        # Fallback for older V2 list formats
                        if isinstance(res, list):
                            for line in res:
                                if isinstance(line, (list, tuple)) and len(line) >= 2:
                                    text_data = line[1]
                                    if isinstance(text_data, (tuple, list)) and len(text_data) >= 2:
                                        texts.append(str(text_data[0]))
                                        scores.append(float(text_data[1]))
                                    elif isinstance(text_data, str):
                                        texts.append(text_data)
                                        scores.append(0.99)

                    # Save valid detections
                    if texts and scores:
                        for text, score in zip(texts, scores):
                            if float(score) > 0.60:
                                batch_extracted.append({
                                    "neighborhood": neighborhood,
                                    "image_id": image_id,
                                    "detected_text": str(text),
                                    "confidence_score": round(float(score), 4)
                                })
            except Exception:
                pass
            
            if len(batch_extracted) >= 50:
                pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
                batch_extracted = []
                gc.collect()

            if i % 100 == 0 or i == total_images:
                print(f"   [{i}/{total_images}] images processed")

        if batch_extracted:
            pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
            gc.collect()
            
        print(f"Finished {neighborhood.title()} - Progress securely written to disk.")

    print(f"\nAll neighborhoods complete. Final dataset intact at: {output_csv}")

if __name__ == "__main__":
    extract_street_typography_gpu()

Resuming run: Found 0 already processed images in CSV.
Initializing OCR Engine (Korean/English Model on GPU)...


C:\Users\barre\anaconda3\envs\zensvi_env\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:686: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\barre\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `C:\Users\barre\.paddlex\official_models\UVDoc`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

RuntimeError: (PreconditionNotMet) op [] kernel output args (0) defs should equal op outputs (1)
  [Hint: Expected op_item->num_results() == output_defs.size(), but received op_item->num_results():1 != output_defs.size():0.] (at ..\paddle\fluid\pir\transforms\pd_op_to_kernel_pass.cc:2468)
